# 9h — Cross-Cluster Term Migration

**Goal:** Track high-value terms across clusters over time. Which cluster 'owns' each term in 2010 vs 2015 vs 2020 vs 2024?

**Finding to demonstrate:** 'transformer' moved NLP→vision→biology. 'diffusion' moved physics→generative AI.

**Output:** `term_migration.json`

**Warning:** Requires loading full TF-IDF matrix (~several GB). Expect 30–60 min compute.

**Prerequisites:** `tfidf_matrix.pkl`, `cluster_labels_500d.pkl`, `arxiv_metadata_features.pkl`, vectorizer pkl for feature names

In [ ]:
import pickle, json, os
import numpy as np
import pandas as pd

DATA = os.path.join(os.path.dirname(os.getcwd()), 'data', 'processed')
OUT  = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'arxiv-trends-website', 'src', 'data')

TARGET_TERMS = [
    'transformer', 'attention', 'diffusion', 'neural', 'quantum',
    'graph', 'contrastive', 'generative', 'embedding', 'latent',
    'optimization', 'gradient', 'stochastic', 'entropy', 'adversarial'
]

TIME_WINDOWS = {
    'pre2015':   (2007, 2014),
    '2015-2019': (2015, 2019),
    '2020-2022': (2020, 2022),
    '2023+':     (2023, 2025)
}

print('Loading cluster labels and metadata...')
with open(os.path.join(DATA, 'cluster_labels_500d.pkl'), 'rb') as f:
    labels = pickle.load(f)
with open(os.path.join(DATA, 'arxiv_metadata_features.pkl'), 'rb') as f:
    meta = pickle.load(f)
print(f'Papers: {len(labels):,}')

In [ ]:
print('Loading TF-IDF matrix (large, slow)...')
with open(os.path.join(DATA, 'tfidf_matrix.pkl'), 'rb') as f:
    tfidf = pickle.load(f)

# Feature names — try multiple naming conventions
feature_names = None
for fname in ['tfidf_feature_names.pkl', 'tfidf_vectorizer.pkl', 'vectorizer.pkl']:
    fpath = os.path.join(DATA, fname)
    if os.path.exists(fpath):
        with open(fpath, 'rb') as f:
            obj = pickle.load(f)
        feature_names = obj.get_feature_names_out() if hasattr(obj, 'get_feature_names_out') else obj
        print(f'Feature names from: {fname} | TF-IDF shape: {tfidf.shape}')
        break

if feature_names is None:
    raise FileNotFoundError('Could not find feature names pkl. Check DATA directory.')

fn_list = list(feature_names)
term_indices = {t: fn_list.index(t) for t in TARGET_TERMS if t in fn_list}
print(f'Found {len(term_indices)}/{len(TARGET_TERMS)} target terms')
print('Missing:', set(TARGET_TERMS) - set(term_indices))

In [ ]:
years  = np.array(meta['year'])
labels_arr = np.array(labels)
results = {}

for term, col_idx in term_indices.items():
    import scipy.sparse as sp
    term_col = np.asarray(tfidf[:, col_idx].todense()).flatten()
    results[term] = {'windows': {}, 'migration': []}
    prev_dominant = None

    for window_name, (yr_start, yr_end) in TIME_WINDOWS.items():
        mask = (years >= yr_start) & (years <= yr_end)
        if mask.sum() == 0: continue

        cluster_scores = {}
        for cid in range(50):
            cmask = mask & (labels_arr == cid)
            if cmask.sum() > 10:
                cluster_scores[cid] = float(term_col[cmask].mean())
        if not cluster_scores: continue

        dominant = max(cluster_scores, key=cluster_scores.get)
        top5 = sorted(cluster_scores.items(), key=lambda x: -x[1])[:5]
        vals = np.array(list(cluster_scores.values())); vals /= vals.sum()
        entropy = float(-np.sum(vals * np.log(vals + 1e-10)))

        results[term]['windows'][window_name] = {
            'dominantCluster': dominant,
            'topClusters': [{'clusterId': int(c), 'score': round(s,6)} for c,s in top5],
            'entropy': round(entropy, 4)
        }

        if prev_dominant is not None and prev_dominant != dominant:
            results[term]['migration'].append({'fromWindow': list(TIME_WINDOWS.keys())[list(TIME_WINDOWS.keys()).index(window_name)-1],
                                               'toWindow': window_name, 'fromCluster': prev_dominant, 'toCluster': dominant})
        prev_dominant = dominant

print('Migration events per term:')
for t, d in results.items():
    if d['migration']: print(f'  {t}: {d["migration"]}')

out_path = os.path.join(OUT, 'term_migration.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved → {out_path}')